# Train PII Baseline — A → B → C pipeline

This notebook implements a small end-to-end pipeline in three steps:

A) `process_a(input)` — load and validate dataset (returns DataFrame)
B) `process_b(a_output)` — train a TF-IDF + One-vs-Rest LogisticRegression multi-label classifier (returns trained pipeline + helper objects)
C) `process_c(b_output)` — evaluate, measure runtime, save artifacts, and (optional) convert to ONNX

Run each cell sequentially. The notebook includes lightweight unit-test-style checks for each step.

In [ ]:
# Section A — process_a: load and validate dataset
import os
import json
import pandas as pd
from typing import Tuple


def process_a(path: str = 'ml/data/synthetic_pii.jsonl') -> pd.DataFrame:
    """
    Load synthetic PII JSONL dataset.

    Args:
        path: Path to JSONL file with records {id, text, labels}.

    Returns:
        DataFrame with columns ['id','text','labels'].

    Raises:
        FileNotFoundError if path does not exist.
        ValueError if dataset is empty or malformed.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found: {path}")

    rows = []
    with open(path, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                # skip bad lines
                continue
            rows.append({
                'id': obj.get('id'),
                'text': obj.get('text',''),
                'labels': obj.get('labels', [])
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError('Loaded dataset is empty')

    return df


# Quick sanity example (unit-test-style check)
if __name__ == '__main__':
    df = process_a('ml/data/synthetic_pii.jsonl')
    print('Loaded rows:', len(df))
    assert 'text' in df.columns and 'labels' in df.columns
    print('Sample:', df.iloc[0].to_dict())


In [ ]:
# Section A — lightweight checks / EDA cells

# Load dataset using process_a
import textwrap

df = process_a('ml/data/synthetic_pii.jsonl')
print('Total rows:', len(df))

# Show a few samples
for i, row in df.head(6).iterrows():
    print('\n--- sample', i)
    print(textwrap.fill(row['text'], 160))
    print('labels =', row['labels'])

# Label distribution
from collections import Counter
all_labels = Counter(l for ls in df['labels'] for l in ls)
print('\nLabel distribution:')
for k, v in all_labels.most_common():
    print(f'  {k}: {v}')


In [ ]:
# Section B — process_b: training the baseline classifier

def process_b(df):
    """
    Train a TF-IDF + One-vs-Rest LogisticRegression multi-label classifier.

    Args:
        df: DataFrame with 'text' and 'labels' columns.

    Returns:
        dict with keys: 'pipeline', 'mlb', 'X_test_texts', 'y_test'
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import MultiLabelBinarizer
    from sklearn.model_selection import train_test_split
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    if 'text' not in df.columns or 'labels' not in df.columns:
        raise ValueError('Input DataFrame must contain text and labels columns')

    texts = df['text'].fillna('').astype(str).tolist()
    labels = df['labels'].apply(lambda x: x if isinstance(x, list) else []).tolist()

    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(labels)

    X_train_texts, X_test_texts, y_train, y_test = train_test_split(
        texts, Y, test_size=0.2, random_state=42
    )

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
        ('clf', OneVsRestClassifier(LogisticRegression(max_iter=1000)))
    ])

    pipeline.fit(X_train_texts, y_train)

    return {
        'pipeline': pipeline,
        'mlb': mlb,
        'X_test_texts': X_test_texts,
        'y_test': y_test
    }

# Unit-check for process_b
if __name__ == '__main__':
    res = process_b(df)
    assert hasattr(res['pipeline'], 'predict')
    assert len(res['mlb'].classes_) > 0
    print('Training completed. Labels:', list(res['mlb'].classes_))


In [ ]:
# Section B — example: run process_b and inspect predictions
res = process_b(df)
pipeline = res['pipeline']
mlb = res['mlb']

# Quick predict on a few examples
sample_texts = df['text'].head(6).tolist()
pred = pipeline.predict(sample_texts)

for t, p in zip(sample_texts, pred.tolist()):
    labels = [mlb.classes_[i] for i, val in enumerate(p) if val == 1]
    print('---')
    print(t)
    print('predicted labels:', labels)


In [ ]:
# Section C — process_c: evaluate, time, save, (optional convert to ONNX)
import time
from sklearn.metrics import classification_report
import joblib
import json
import os


def process_c(b_output, model_dir='ml/models'):
    """
    Evaluate pipeline on test set, measure runtime, save artifacts, and attempt ONNX export if available.

    Args:
        b_output: dict returned by process_b
        model_dir: output directory

    Returns:
        dict with evaluation metrics and runtime info
    """
    pipeline = b_output['pipeline']
    mlb = b_output['mlb']
    X_test = b_output['X_test_texts']
    y_test = b_output['y_test']

    os.makedirs(model_dir, exist_ok=True)

    t0 = time.time()
    y_pred = pipeline.predict(X_test)
    t1 = time.time()

    elapsed = t1 - t0

    report = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)

    # Save artifacts
    vec_path = os.path.join(model_dir, 'pipeline.joblib')
    classes_path = os.path.join(model_dir, 'classes.json')

    joblib.dump(pipeline, vec_path)
    with open(classes_path, 'w', encoding='utf-8') as fh:
        json.dump(list(mlb.classes_), fh)

    result = {
        'elapsed_seconds': elapsed,
        'report': report,
        'pipeline_path': vec_path,
        'classes_path': classes_path
    }

    # Optional: try to export to ONNX if skl2onnx available
    try:
        from skl2onnx import convert_sklearn
        from skl2onnx.common.data_types import StringTensorType
        onnx_path = os.path.join(model_dir, 'pii_classifier.onnx')
        initial_types = [('input', StringTensorType([None, 1]))]
        onx = convert_sklearn(pipeline, initial_types=initial_types)
        with open(onnx_path, 'wb') as f:
            f.write(onx.SerializeToString())
        result['onnx_path'] = onnx_path
    except Exception as e:
        result['onnx_error'] = str(e)

    return result


# Run full A -> B -> C pipeline here
if __name__ == '__main__':
    a = process_a('ml/data/synthetic_pii.jsonl')
    b = process_b(a)
    c = process_c(b)
    print('Elapsed prediction time (s):', c['elapsed_seconds'])
    if 'onnx_path' in c:
        print('ONNX saved:', c['onnx_path'])
    else:
        print('ONNX export not available:', c.get('onnx_error'))
    print('Saved pipeline:', c['pipeline_path'])


In [ ]:
# Integration test cell (end-to-end smoke check)

a = process_a('ml/data/synthetic_pii.jsonl')
print('A rows:', len(a))

b = process_b(a)
print('Trained on labels:', list(b['mlb'].classes_))

c = process_c(b)
print('Elapsed (s):', c['elapsed_seconds'])

# Simple assertion: at least one label should have non-zero recall in report
reports = c['report']
any_recall = any(reports[label]['recall'] > 0 for label in reports if label in b['mlb'].classes_)
assert any_recall, 'No label had positive recall — something is wrong'
print('Integration test passed')
